# Notebook 14 — Inference-Time / Computational Cost Benchmark
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Measure actual per-game inference latency for Model D (structured-only) versus the
full SBERT-encode + PCA + Model E path, then combine this with the already-known 23.4% Model E
invocation rate (from NB04) to produce a concrete compute-savings statement. This answers
Reviewer 1's comment: *"the paper should report how often each specialist is activated and the
computational savings compared with always using the highest-tier model."*

The activation rate (23.4%) is already established in NB04 — this notebook adds the missing
half: actual wall-clock timing.

**This is a READ-ONLY analysis notebook.** It loads already-saved models and times inference
calls only. It does not retrain anything and does not modify any file from NB01–NB13.

**Inputs (read-only):**
- `data/processed/games_features_t4.csv` (for realistic T4 feature vectors to time CatBoost on)
- `models/model_d.pkl`, `models/model_e.pkl`, `models/pca_transformer.pkl`

**No raw dataset access needed.** For timing the SBERT encoding step, this notebook uses
synthetic placeholder text of realistic length (matching typical Steam short-description length),
since encoding *speed* depends on text length and batch size, not on the actual semantic content
— we are not testing accuracy here, only latency.

**A note on hardware:** NB05's SBERT encoding was run on Colab with a T4 GPU ("~2 min GPU vs.
~35 min CPU" for encoding all 20,383 descriptions). This notebook is written to run on whatever
hardware is available in your local `fenv` environment. If that is CPU-only, the numbers below
are arguably *more* representative of a real deployment scenario (e.g. a Streamlit app running on
a normal server without a GPU) than the Colab training-time numbers, since GPUs are rarely used
just to serve single predictions in production.

---
## Contents
1. Setup and Imports
2. Load Saved Models
3. Load a Sample of Real T4 Feature Vectors (for timing Model D)
4. Generate Synthetic Description Text (for timing SBERT — content-agnostic)
5. Time Each Pipeline Stage
6. Project Total Cost: Direct Routing vs. Always-Model-E vs. Actual Uncertainty Router
7. Summary


## 1. Setup and Imports

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TARGET_COL, RANDOM_STATE, FEATURES_T4_CSV, TIER_FEATURES,
    MODEL_PATHS, PCA_PATH,
)

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    print('sentence-transformers not installed. Install with:')
    print('  pip install sentence-transformers --break-system-packages')
    SBERT_AVAILABLE = False

print(f'Project root : {PROJECT_ROOT}')
print(f'Random state : {RANDOM_STATE}')
print(f'sentence-transformers available: {SBERT_AVAILABLE}')


Project root : C:\Users\3214h\Documents\fyp-steam-reception
Random state : 42
sentence-transformers available: True


## 2. Load Saved Models

Loading is timed separately from inference, since model/library load time is a one-off startup
cost in a real deployment (e.g. a Streamlit app loads models once when it starts, not once per
prediction), whereas per-game inference time is paid on every single request.


In [2]:
# Uses MODEL_PATHS and PCA_PATH imported from config.py in Cell 2 —
# no local path redefinition, one source of truth.
assert MODEL_PATHS['D'].exists(), f"Not found: {MODEL_PATHS['D']}"
assert MODEL_PATHS['E'].exists(), f"Not found: {MODEL_PATHS['E']}"
assert PCA_PATH.exists(), f'Not found: {PCA_PATH}'

t0 = time.time()
model_d = joblib.load(MODEL_PATHS['D'])
load_time_d = time.time() - t0
print(f'Model D loaded in {load_time_d:.3f}s')

t0 = time.time()
model_e = joblib.load(MODEL_PATHS['E'])
load_time_e = time.time() - t0
print(f'Model E loaded in {load_time_e:.3f}s')

t0 = time.time()
pca = joblib.load(PCA_PATH)
load_time_pca = time.time() - t0
print(f'PCA transformer loaded in {load_time_pca:.3f}s')

sbert_model = None
if SBERT_AVAILABLE:
    t0 = time.time()
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    load_time_sbert = time.time() - t0
    print(f'SBERT model (all-MiniLM-L6-v2) loaded in {load_time_sbert:.3f}s')
    device = 'cuda' if str(sbert_model.device).startswith('cuda') else 'cpu'
    print(f'SBERT running on: {device}')
else:
    print('Skipping SBERT load — library not available.')


Model D loaded in 0.170s
Model E loaded in 0.008s
PCA transformer loaded in 0.106s


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT model (all-MiniLM-L6-v2) loaded in 9.191s
SBERT running on: cpu


## 3. Load a Sample of Real T4 Feature Vectors

We use real (already-processed) T4 feature vectors from `games_features_t4.csv` so the CatBoost
timing reflects the actual feature dimensionality (53 features) used in production, not a
synthetic stand-in.


In [3]:
assert FEATURES_T4_CSV.exists(), f'Not found: {FEATURES_T4_CSV}'

df = pd.read_csv(FEATURES_T4_CSV)

# Uses config.py's verified TIER_FEATURES[4] — NOT df.columns directly, which
# includes 2 extra non-feature columns (dlc_count, short_description) that
# would crash predict_proba() or corrupt the timing (see chat history).
feature_cols = TIER_FEATURES[4]
missing = [c for c in feature_cols if c not in df.columns]
assert not missing, f'Columns missing from CSV: {missing}'

N_SAMPLE = 200   # number of games to time — enough for a stable per-game average
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(df), size=N_SAMPLE, replace=False)
X_sample = df.iloc[sample_idx][feature_cols].values

print(f'Sampled {N_SAMPLE} real T4 feature vectors ({X_sample.shape[1]} features each) for timing.')


Sampled 200 real T4 feature vectors (53 features each) for timing.


## 4. Generate Synthetic Description Text

Content is irrelevant for a timing benchmark — SBERT's encoding speed depends on token count and
batch size, not semantics. These strings match the typical length of a Steam short description
(roughly one to three sentences).


In [4]:
SYNTHETIC_DESCRIPTIONS = [
    "A fast-paced roguelike dungeon crawler with procedurally generated levels, permadeath, "
    "and hundreds of unique item combinations to discover on every run.",
    "Explore a vast open world as a lone survivor scavenging for resources, building shelter, "
    "and defending against waves of hostile creatures each night.",
    "A cozy farming simulator where you restore an abandoned homestead, befriend the local "
    "townsfolk, and grow crops through four seasons of gameplay.",
    "Command your fleet in this turn-based tactical strategy game set across a war-torn galaxy "
    "with branching campaign choices and permanent consequences.",
    "A narrative-driven puzzle adventure following two siblings as they solve environmental "
    "puzzles together to escape a mysterious, ever-shifting mansion.",
] * (200 // 5)   # repeat to get N_SAMPLE total, matching the T4 sample size above

SYNTHETIC_DESCRIPTIONS = SYNTHETIC_DESCRIPTIONS[:N_SAMPLE]
print(f'Generated {len(SYNTHETIC_DESCRIPTIONS)} synthetic description strings for timing.')
print(f'Example: "{SYNTHETIC_DESCRIPTIONS[0][:80]}..."')


Generated 200 synthetic description strings for timing.
Example: "A fast-paced roguelike dungeon crawler with procedurally generated levels, perma..."


## 5. Time Each Pipeline Stage

We time three things separately:
1. **Model D predict** — structured features only (the always-run path)
2. **SBERT encode** — one description at a time (matches real-time single-prediction deployment,
   not the large-batch encoding NB05 did once for the whole dataset)
3. **Model E predict** — after PCA transform, on the concatenated T4 + text-embedding vector

A warm-up call is discarded before timing starts, since the first call to any of these often
includes one-off JIT/lazy-loading overhead that would not repeat on subsequent calls.


In [5]:
# ── Warm-up (discarded) ────────────────────────────────────────────────────────
_ = model_d.predict_proba(X_sample[0:1])
if sbert_model is not None:
    _ = sbert_model.encode([SYNTHETIC_DESCRIPTIONS[0]])

# ── Time Model D predict, one game at a time ────────────────────────────────────
times_d = []
for i in range(N_SAMPLE):
    t0 = time.perf_counter()
    model_d.predict_proba(X_sample[i:i+1])
    times_d.append(time.perf_counter() - t0)
times_d = np.array(times_d) * 1000   # convert to milliseconds

print(f'Model D predict (per game): mean={times_d.mean():.3f}ms  std={times_d.std():.3f}ms')


Model D predict (per game): mean=1.078ms  std=0.214ms


In [6]:
if sbert_model is not None:
    # ── Time SBERT encode, one description at a time ────────────────────────────
    times_sbert = []
    for desc in SYNTHETIC_DESCRIPTIONS:
        t0 = time.perf_counter()
        emb = sbert_model.encode([desc])
        times_sbert.append(time.perf_counter() - t0)
    times_sbert = np.array(times_sbert) * 1000

    print(f'SBERT encode (per game): mean={times_sbert.mean():.3f}ms  std={times_sbert.std():.3f}ms')

    # ── Time PCA transform (negligible — simple matrix multiply) ────────────────
    raw_embeddings = sbert_model.encode(SYNTHETIC_DESCRIPTIONS)
    times_pca = []
    for i in range(N_SAMPLE):
        t0 = time.perf_counter()
        pca.transform(raw_embeddings[i:i+1])
        times_pca.append(time.perf_counter() - t0)
    times_pca = np.array(times_pca) * 1000

    print(f'PCA transform (per game): mean={times_pca.mean():.4f}ms  std={times_pca.std():.4f}ms')

    # ── Time Model E predict on the concatenated T4 + PCA-text vector ───────────
    embeddings_pca = pca.transform(raw_embeddings)
    X_t5_sample = np.hstack([X_sample, embeddings_pca])

    times_e = []
    for i in range(N_SAMPLE):
        t0 = time.perf_counter()
        model_e.predict_proba(X_t5_sample[i:i+1])
        times_e.append(time.perf_counter() - t0)
    times_e = np.array(times_e) * 1000

    print(f'Model E predict (per game): mean={times_e.mean():.3f}ms  std={times_e.std():.3f}ms')

    # ── Full Model E path = SBERT + PCA + Model E predict ───────────────────────
    full_e_path_per_game = times_sbert.mean() + times_pca.mean() + times_e.mean()
    print()
    print(f'Full Model E path (SBERT + PCA + predict), per game: {full_e_path_per_game:.3f}ms')
else:
    print('Skipping SBERT/PCA/Model E timing — sentence-transformers not available.')


SBERT encode (per game): mean=19.824ms  std=4.624ms
PCA transform (per game): mean=0.2924ms  std=2.0231ms
Model E predict (per game): mean=0.897ms  std=0.240ms

Full Model E path (SBERT + PCA + predict), per game: 21.014ms


## 6. Project Total Cost: Direct Routing vs. Always-Model-E vs. Actual Uncertainty Router

Using the per-game timings above and the known invocation rate from NB04 (Model E consulted for
23.4% of the 4,077-game test set), project the total inference time under three strategies:

- **Direct routing (Model D only):** every game gets only the cheap structured-feature path.
- **Always run Model E:** every game gets the full SBERT+PCA+Model E path (the worst case the
  reviewer is asking us to compare against — i.e. "always using the highest-tier model").
- **Actual uncertainty router:** every game gets Model D (cheap), and only 23.4% of games
  additionally get the expensive Model E path.


In [7]:
if sbert_model is not None:
    N_TEST = 4077   # test set size from NB04
    PCT_MODEL_E = 0.234   # Model E invocation rate from NB04

    cost_direct_only   = N_TEST * times_d.mean()
    cost_always_e       = N_TEST * full_e_path_per_game
    cost_actual_router  = N_TEST * times_d.mean() + (N_TEST * PCT_MODEL_E) * full_e_path_per_game

    print('=== PROJECTED TOTAL INFERENCE TIME (test set, N=4,077 games) ===')
    print(f'  Direct routing (Model D only)      : {cost_direct_only/1000:.2f}s total  '
          f'({times_d.mean():.3f}ms/game)')
    print(f'  Always run Model E (worst case)    : {cost_always_e/1000:.2f}s total  '
          f'({full_e_path_per_game:.3f}ms/game)')
    print(f'  Actual uncertainty router (23.4%)  : {cost_actual_router/1000:.2f}s total  '
          f'({cost_actual_router/N_TEST:.3f}ms/game average)')
    print()

    savings_pct = (1 - cost_actual_router / cost_always_e) * 100
    print(f'Compute saved by selective routing vs. always running Model E: {savings_pct:.1f}%')
    print()
    print('(For reference — NB05 reported SBERT encoding all 20,383 descriptions once, in bulk, '
          'takes ~2 min on a Colab T4 GPU vs. ~35 min on CPU. The per-game numbers above reflect '
          'single-prediction latency, the relevant number for a live Streamlit deployment, not '
          'bulk one-time encoding.)')
else:
    print('Skipping cost projection — requires SBERT timing from Section 5.')


=== PROJECTED TOTAL INFERENCE TIME (test set, N=4,077 games) ===
  Direct routing (Model D only)      : 4.40s total  (1.078ms/game)
  Always run Model E (worst case)    : 85.67s total  (21.014ms/game)
  Actual uncertainty router (23.4%)  : 24.44s total  (5.995ms/game average)

Compute saved by selective routing vs. always running Model E: 71.5%

(For reference — NB05 reported SBERT encoding all 20,383 descriptions once, in bulk, takes ~2 min on a Colab T4 GPU vs. ~35 min on CPU. The per-game numbers above reflect single-prediction latency, the relevant number for a live Streamlit deployment, not bulk one-time encoding.)


## 7. Summary

Copy the compute-savings percentage and per-game latency numbers above directly into the
Discussion section, alongside the existing 23.4% activation-rate statement from NB04. Suggested
wording: *"Selective invocation of Model E reduces total inference cost by [X]% relative to
always consulting the highest-tier model, since the SBERT encoding step ([Y]ms/game) dominates
Model E's cost and is only paid for the 23.4% of games the router actually flags as uncertain."*

If `sentence-transformers` was not available in this environment, install it and re-run Sections
2, 5, and 6 before writing this into the paper — Sections 1–4 will have run regardless.

Nothing in NB01–NB13 was modified by running this notebook.
